# 05 — Making a Forecast

**Goal:** actually use the models. Notebook 04 decided *which* model to keep for each site and
indicator; this notebook turns that decision into a real **4-hour-ahead forecast**.

All the logic lives in `forecast.py`, so the same code can be used outside Jupyter — from a
script, a scheduled job, or a dashboard.

**What you need first:**

| Needed | Produced by |
|---|---|
| `processed/*.pkl` | `02_data_preprocessing.ipynb` |
| `models/*.pkl` | `03_model_training.ipynb` |
| `results/final_decision.csv` | `04_evaluation_comparison.ipynb` |

## Step 1 — Load the forecasting code

In [1]:
import warnings

import pandas as pd

from forecast import forecast, forecast_all, chosen_model, SITE_NAME, HORIZON_HOURS

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)

print("Forecast horizon:", HORIZON_HOURS, "hours")
print("Sites:", list(SITE_NAME.values()))

Forecast horizon: 4 hours
Sites: ['Bay Mau', 'Cau Nga', 'Ho Tay']


## Step 2 — Which model will be used for each case?

`forecast.py` reads `results/final_decision.csv`, so it always uses whatever notebook 04 last
decided. If you rerun the comparison and the decision changes, forecasting follows
automatically — there is no model name hard-coded anywhere.

In [2]:
decision = pd.read_csv("results/final_decision.csv")
decision[["site", "indicator", "keep_this_model", "skill", "reason"]].round(3)

,site,indicator,keep_this_model,skill,reason
0,Bay Mau,nh4,persistence,0.000,nothing beat the baseline
1,Bay Mau,cod,persistence,0.000,nothing beat the baseline
2,Bay Mau,tss,persistence,0.000,nothing beat the baseline
3,Cau Nga,nh4,persistence,0.000,nothing beat the baseline
4,Cau Nga,cod,persistence,0.000,nothing beat the baseline
5,Cau Nga,tss,gradient_boosting,0.194,proven better than the baseline
6,Ho Tay,nh4,persistence,0.000,nothing beat the baseline
7,Ho Tay,cod,persistence,0.000,nothing beat the baseline
8,Ho Tay,tss,moving_average,0.173,proven better than the baseline


Notice how few real models there are. `persistence` and `moving_average` have **no trained
parameters at all** — they are rules, recreated in one line each. Only `gradient_boosting`
loads a file from `models/`.

That is the practical consequence of the baseline winning: there is barely anything to deploy.

## Step 3 — Load the most recent readings

In production this would be the last day or two of live sensor data. Here we use the end of the
2024 record. 48 hours is more than any model needs — the largest requirement is 25 hours.

In [3]:
recent_readings = {}
for site in SITE_NAME:
    hourly = pd.read_pickle("processed/" + site + "_hourly.pkl")["values"]
    recent_readings[site] = hourly.tail(48)

print("Most recent hour available:", recent_readings["CAU_NGA"].index[-1])
print()
recent_readings["CAU_NGA"][["nh4", "cod", "tss"]].tail(6)

Most recent hour available: 2024-12-31 23:00:00



,nh4,cod,tss
datetime,,,
2024-12-31 18:00:00,0.719167,15.733333,30.924167
2024-12-31 19:00:00,0.726667,15.993333,30.968333
2024-12-31 20:00:00,0.738333,16.170000,31.035000
2024-12-31 21:00:00,0.741667,15.696667,29.957500
2024-12-31 22:00:00,0.750000,15.423333,28.024167
2024-12-31 23:00:00,0.755833,15.354167,27.017500


## Step 4 — Make a single forecast

Start with one case so it is clear what happens. Cau Nga's TSS is the one place where a
machine-learning model was proven better, so it loads the saved gradient boosting model.

In [4]:
result = forecast("CAU_NGA", "tss", recent_readings["CAU_NGA"])

for key, value in result.items():
    print(f"{key:20s} {value}")

site                 Cau Nga
indicator            tss
forecast             16.93439439515298
forecast_from        2024-12-31 23:00:00
valid_for            2025-01-01 03:00:00
model_used           gradient_boosting
last_reading_at      2024-12-31 23:00:00
reading_age_hours    0.0


The result also reports `reading_age_hours`: how old the most recent genuine measurement is.
A forecast built on a reading from six hours ago is far less trustworthy than one built on a
reading from the current hour, and the sensors at these sites drop out often, so this is worth
watching.

## Step 5 — Forecast every indicator at every site

In [5]:
all_forecasts = forecast_all(recent_readings)
all_forecasts.round(3)

,site,indicator,forecast,forecast_from,valid_for,model_used,last_reading_at,reading_age_hours
0,Bay Mau,nh4,1.160,2024-12-31 23:00:00,2025-01-01 03:00:00,persistence,2024-12-31 23:00:00,0.0
1,Bay Mau,cod,15.150,2024-12-31 23:00:00,2025-01-01 03:00:00,persistence,2024-12-31 22:00:00,1.0
2,Bay Mau,tss,9.544,2024-12-31 23:00:00,2025-01-01 03:00:00,persistence,2024-12-31 23:00:00,0.0
3,Cau Nga,nh4,0.756,2024-12-31 23:00:00,2025-01-01 03:00:00,persistence,2024-12-31 23:00:00,0.0
4,Cau Nga,cod,15.354,2024-12-31 23:00:00,2025-01-01 03:00:00,persistence,2024-12-31 23:00:00,0.0
5,Cau Nga,tss,16.934,2024-12-31 23:00:00,2025-01-01 03:00:00,gradient_boosting,2024-12-31 23:00:00,0.0
6,Ho Tay,nh4,1.370,2024-12-31 23:00:00,2025-01-01 03:00:00,persistence,2024-12-31 23:00:00,0.0
7,Ho Tay,cod,9.517,2024-12-31 23:00:00,2025-01-01 03:00:00,persistence,2024-12-31 23:00:00,0.0
8,Ho Tay,tss,4.685,2024-12-31 23:00:00,2025-01-01 03:00:00,moving_average,2024-12-31 23:00:00,0.0


In [6]:
print("How many forecasts come from each model?")
print()
for name, count in all_forecasts["model_used"].value_counts().items():
    print("   ", name, "->", count, "of", len(all_forecasts))

stale = all_forecasts[all_forecasts["reading_age_hours"] > 0]
print()
if len(stale) > 0:
    print("Forecasts built on an older reading than the current hour:")
    display(stale[["site", "indicator", "last_reading_at", "reading_age_hours"]])
else:
    print("Every forecast used a reading from the current hour.")

How many forecasts come from each model?

    persistence -> 7 of 9
    gradient_boosting -> 1 of 9
    moving_average -> 1 of 9

Forecasts built on an older reading than the current hour:


,site,indicator,last_reading_at,reading_age_hours
1,Bay Mau,cod,2024-12-31 22:00:00,1.0


## Step 6 — What if we had used a different model?

`forecast()` accepts a `model_name` argument, which lets you see what a different choice would
have produced for the same moment. This is a useful sanity check, and it shows concretely how
little separates the models when the baseline wins.

In [7]:
comparison = []
for model_name in ["persistence", "moving_average", "drift", "ridge", "gradient_boosting"]:
    try:
        answer = forecast("CAU_NGA", "tss", recent_readings["CAU_NGA"],
                          model_name=model_name)
        comparison.append({"model": model_name, "forecast": answer["forecast"]})
    except (FileNotFoundError, ValueError) as problem:
        comparison.append({"model": model_name, "forecast": None,
                           "note": str(problem)[:60]})

print("Cau Nga TSS, 4 hours ahead, according to each model:")
pd.DataFrame(comparison).round(3)

Cau Nga TSS, 4 hours ahead, according to each model:


,model,forecast
0,persistence,27.017
1,moving_average,29.009
2,drift,23.067
3,ridge,21.356
4,gradient_boosting,16.934


## Step 7 — Checking the forecasts are correct

An easy mistake when moving from a notebook to a reusable script is to build the input features
slightly differently, which silently changes the answer. The check below re-forecasts hours we
already have predictions for, and confirms `forecast.py` reproduces notebook 03 exactly.

In [8]:
import numpy as np

print("Comparing forecast.py against the predictions saved by notebook 03")
print()

for site, indicator, model_name in [("CAU_NGA", "tss", "gradient_boosting"),
                                    ("HO_TAY", "tss", "moving_average"),
                                    ("BAY_MAU", "nh4", "persistence")]:
    saved = pd.read_csv("predictions/" + site + "_" + indicator + ".csv",
                        index_col=0, parse_dates=True)
    hourly = pd.read_pickle("processed/" + site + "_hourly.pkl")["values"]

    differences = []
    for moment in saved.index[-25:]:
        again = forecast(site, indicator, hourly.loc[:moment], model_name=model_name)
        differences.append(abs(again["forecast"] - saved.loc[moment, model_name]))

    biggest = max(differences)
    verdict = "MATCH" if biggest < 1e-6 else "MISMATCH - investigate"
    print(f"   {verdict:9s} {SITE_NAME[site]:8s} {indicator:4s} "
          f"{model_name:18s} largest difference = {biggest:.2e}")

Comparing forecast.py against the predictions saved by notebook 03



   MATCH     Cau Nga  tss  gradient_boosting  largest difference = 3.55e-15
   MATCH     Ho Tay   tss  moving_average     largest difference = 1.78e-15
   MATCH     Bay Mau  nh4  persistence        largest difference = 2.22e-16


## Using this outside Jupyter

`forecast.py` runs on its own:

```bash
python forecast.py
```

Or import it into your own code:

```python
import pandas as pd
from forecast import forecast

recent = pd.read_pickle("processed/CAU_NGA_hourly.pkl")["values"].tail(48)
answer = forecast("CAU_NGA", "tss", recent)

print(answer["forecast"], "expected at", answer["valid_for"])
```

`recent` must be an **hourly** table with a DatetimeIndex, ending at the moment you are
forecasting from, and containing the columns that site needs (see
`processed/selected_features.json`).

### Things to be careful about

1. **Watch `reading_age_hours`.** If the newest genuine reading is several hours old, the
   forecast is weaker than it looks. These sensors go offline often.
2. **Feed it hourly data.** Raw 5-minute readings must be averaged into hours first, exactly as
   notebook 02 does, or the lag features will mean something different.
3. **Retrain occasionally.** The gradient boosting model learned from 2024. If the water
   changes, rerun notebooks 03 and 04 — and the decision itself may change, which
   `forecast.py` will pick up automatically.
4. **The baseline is not a placeholder.** For 7 of 9 cases it *is* the answer, and it was
   chosen because nothing beat it — not because nothing else was tried.